# render

> Put a demo on a page: the bundle, a static page, `show` for notebooks, `md` for Markdown, and `Demo` for FastHTML.

One runtime draws a spec everywhere. A host element carries the spec as inert JSON, and the runtime mounts it:

```html
<div data-rupaka><script type="application/json">{...spec...}</script></div>
```

The runtime boots on load and after each htmx swap. Nothing is templated into the script.

In [ ]:
#| default_exp render

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import base64, html, json, shutil
from pathlib import Path
from fastcore.utils import *
from fastcore.xml import Div, Script, Link, Safe, to_xml
from rupaka.spec import PKG, manifests

## The bundle

`bundle` joins the core, the fns, the blocks and the boot script in a fixed order. There is no bundler and no build step.

In [ ]:
#| export
STATIC = PKG/'static'

def assets() -> AttrDict:
    "The files of the runtime, in bundle order."
    blocks = sorted((PKG/'blocks').glob('*/'))
    return AttrDict(js=[STATIC/'rupaka.js', *sorted((PKG/'fns').glob('*.js')), *[d/f'{d.name}.js' for d in blocks], STATIC/'boot.js'],
                    css=[STATIC/'rupaka.css', *[d/f'{d.name}.css' for d in blocks]], page=STATIC/'page.css', fonts=STATIC/'fonts')

def bundle(kind:str='js') -> str:
    "The runtime as one script (`js`) or one style sheet (`css`)."
    return '\n'.join(f'/* {p.relative_to(PKG)} */\n{p.read_text()}' for p in assets()[kind])

The script starts with the core and ends with the boot. Each block is in it.

In [ ]:
js = bundle()
assert js.startswith('/* static/rupaka.js */') and js.rstrip().endswith('})();')
for n in manifests(): assert f"Rupaka.block('{n}'" in js, n
assert 'eval(' not in js and 'new Function' not in js and 'fetch(' not in js

Each block style uses only the `--rk-*` vars for colour, so a demo takes the look of the page that holds it.

In [ ]:
import re

In [ ]:
for p in assets().css[1:]:
    cols = re.findall(r'(?:#[0-9a-fA-F]{3,6}\b|rgba?\(|hsl\()', p.read_text())
    test_eq((p.name, cols), (p.name, []))

## The host element

`host` writes the element that carries a spec. JSON in a `<script>` must not hold `</script>`, so every `<` becomes `\u003c`. `JSON.parse` reads it back as `<`.

In [ ]:
#| export
def spec_json(spec) -> str:
    "The spec as JSON that is safe inside a `<script>` element."
    return json.dumps(spec, ensure_ascii=False).replace('<', '\\u003c')

def host(spec) -> str:
    "The host element: a div that carries its spec as inert JSON."
    return f'<div data-rupaka><script type="application/json">{spec_json(spec)}</script></div>'

In [ ]:
h = host(dict(id='x', ask='Do not stop at </script> here.'))
assert '</script>' not in h.split('<script type="application/json">')[1][:-len('</script></div>')]
test_eq(json.loads(h.split('json">')[1].split('</script>')[0]), dict(id='x', ask='Do not stop at </script> here.'))

## A static page

`page` writes one self-contained HTML file: the runtime, the style, the specs, and abhyasa's palette. The `fonts` argument picks how the page gets its fonts:

| `fonts` | what it does |
|---|---|
| `'link'` | links Literata and Libre Baskerville from `./fonts` (`save` copies them) |
| `'embed'` | puts the fonts in the page as data URIs, about 160 KB more |
| `'none'` | uses the system serif; for a sandboxed `srcdoc` frame, which cannot load files |

In [ ]:
#| export
FONTS = [('Literata', 'normal', '200 600', 'literata-latin-wght-normal.woff2', 'U+0000-00FF,U+2000-206F,U+2190-2193,U+2212'),
         ('Literata', 'normal', '200 600', 'literata-latin-ext-wght-normal.woff2', 'U+0100-02BA,U+02BD-02C5,U+02C7-02CC,U+02CE-02D7,U+02DD-02FF,U+0304,U+0308,U+1E00-1E9F'),
         ('Literata', 'italic', '200 600', 'literata-latin-wght-italic.woff2', 'U+0000-00FF,U+2000-206F'),
         ('Literata', 'italic', '200 600', 'literata-latin-ext-wght-italic.woff2', 'U+0100-02BA,U+1E00-1E9F'),
         ('Libre Baskerville', 'normal', '400', 'libre-baskerville-latin-400-normal.woff2', 'U+0000-00FF,U+2000-206F'),
         ('Libre Baskerville', 'normal', '400', 'libre-baskerville-latin-ext-400-normal.woff2', 'U+0100-02BA,U+1E00-1E9F')]

def font_css(fonts:str='link', base:str='fonts/') -> str:
    "The `@font-face` rules: linked from `base`, embedded as data URIs, or none."
    if fonts == 'none': return ''
    def src(f):
        if fonts == 'link': return f'{base}{f}'
        return 'data:font/woff2;base64,' + base64.b64encode((STATIC/'fonts'/f).read_bytes()).decode()
    return '\n'.join(f"@font-face {{ font-family: '{n}'; font-style: {s}; font-weight: {w}; font-display: swap; src: url({src(f)}) format('woff2'); unicode-range: {u}; }}"
                     for n, s, w, f, u in FONTS)

`page` takes one spec or a list of specs. A list makes a gallery page with a rule between the demos.

In [ ]:
#| export
def page(specs, title:str=None, fonts:str='link', lead:str='') -> str:
    "One self-contained HTML document that draws `specs`, with no network."
    specs = listify(specs) if not isinstance(specs, dict) else [specs]
    title = title or specs[0].get('title') or specs[0].get('id', 'rupaka')
    body = '<hr class="rk-sect">'.join(host(s) for s in specs)
    head = f'<h1 class="rk-h1">{html.escape(title)}</h1><p class="rk-lead">{html.escape(lead)}</p>' if len(specs) > 1 else ''
    css = '\n'.join([font_css(fonts), assets().page.read_text(), bundle('css'),
                     '.rk-h1 { font: 400 30px/1.3 var(--head); margin: 0 0 6px; } .rk-lead { color: var(--mute); margin: 0 0 8px; font-size: 16px; }',
                     'hr.rk-sect { border: 0; border-top: 1px solid var(--line); margin: 40px 0 24px; }'])
    return (f'<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
            f'<title>{html.escape(title)}</title><style>{css}</style></head><body><main class="rk-page">{head}{body}</main>'
            f'<script>{bundle()}</script></body></html>')

A page loads nothing from the network: no `src`, `href` or `url()` points at `http`. (The SVG namespace string is not a request.) With `fonts='none'` it has no `@font-face` rule.

In [ ]:
spec = dict(rupaka=1, id='t', title='T', state=dict(a=1), blocks=[dict(type='readout', text='{a}')])
p = page(spec)
assert not re.search(r'''(src|href)=["']?https?:|url\(\s*["']?https?:''', p)
assert '@font-face' in p and '@font-face' not in page(spec, fonts='none')
assert 'data:font/woff2;base64,' in page(spec, fonts='embed')

`save` writes a page to a file. With linked fonts, it copies the font files beside it, so the file works from `file://`.

In [ ]:
#| export
def save(specs, path, fonts:str='link', **kw) -> Path:
    "Write `page(specs)` to `path`; with `fonts='link'`, copy the fonts to `fonts/` beside it."
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(page(specs, fonts=fonts, **kw))
    if fonts == 'link': shutil.copytree(STATIC/'fonts', p.parent/'fonts', dirs_exist_ok=True)
    return p

In [ ]:
import tempfile

In [ ]:
out = save(spec, Path(tempfile.mkdtemp())/'t.html')
assert (out.parent/'fonts'/'literata-latin-wght-normal.woff2').exists()

## Notebooks: show

`show(spec)` returns an object that a notebook draws. It gives two mime types:

- `application/vnd.rupaka+json`: the spec, for a frontend that knows rupaka (a later leela change).
- `text/html`: the whole page with `fonts='none'`. leela, Jupyter and VS Code draw it in a sandboxed frame. This works in leela today.

In [ ]:
#| export
MIME = 'application/vnd.rupaka+json'

class Show:
    "A demo for a notebook output: the rupaka mime and a full HTML page."
    def __init__(self, spec): self.spec = spec
    def _repr_html_(self): return page(self.spec, fonts='none')
    def _repr_mimebundle_(self, include=None, exclude=None): return {MIME: self.spec, 'text/html': self._repr_html_()}
    def __repr__(self): return f"Show({self.spec.get('id')!r})"

def show(spec) -> Show:
    "Draw `spec` in a notebook output."
    return Show(spec)

In [ ]:
mb = show(spec)._repr_mimebundle_()
test_eq(mb[MIME], spec)
assert '<script' in mb['text/html'] and '@font-face' not in mb['text/html']

## Markdown: md

`md(spec)` writes a ```` ```rupaka ```` fence with readable JSON. A renderer that knows rupaka mounts it; others show the JSON as code.
`md(spec, frame=True)` writes a ```` ```{=html} ```` fence that holds a sandboxed frame. leela draws this in a Markdown cell today, with no change to leela.

In [ ]:
#| export
def md(spec, frame:bool=False, height:int=760) -> str:
    "A Markdown fence for `spec`: readable JSON, or a sandboxed frame."
    if not frame: return f"```rupaka\n{json.dumps(spec, ensure_ascii=False, indent=1)}\n```"
    src = html.escape(page(spec, fonts='none'), quote=True)
    return f'```{{=html}}\n<iframe class="rk-frame" sandbox="allow-scripts" style="width:100%;height:{height}px;border:0" srcdoc="{src}"></iframe>\n```'

In [ ]:
assert md(spec).startswith('```rupaka\n{') and json.loads(md(spec)[10:-4]) == spec
f = md(spec, frame=True)
assert f.startswith('```{=html}\n<iframe') and '<script' not in f and '&lt;script' in f

## FastHTML: Demo and Assets

`Demo(spec)` is the host element as an FT component. `Assets(prefix)` gives the `Link` and the `Script` for a server that serves the bundle at `prefix`.

In [ ]:
#| export
def Demo(spec, **kw):
    "The host element of `spec` for a FastHTML page."
    return Div(Script(Safe(spec_json(spec)), type='application/json'), data_rupaka='', **kw)

def Assets(prefix:str='/rupaka', v:str=''):
    "The style and the script of the runtime, served at `prefix`; `v` busts the cache."
    q = f'?v={v}' if v else ''
    return Link(rel='stylesheet', href=f'{prefix}/rupaka.css{q}'), Script(src=f'{prefix}/rupaka.js{q}')

In [ ]:
x = to_xml(Demo(dict(id='d', ask='a </script> b')))
assert '<script type="application/json">' in x and 'data-rupaka' in x and 'a \\u003c/script> b' in x
test_eq(to_xml(Assets(v='1')[1]), '<script src="/rupaka/rupaka.js?v=1"></script>')

`version()` is a short hash of the bundle, for a cache-busting `v`.

In [ ]:
#| export
def version() -> str:
    "A short hash of the script and the style."
    import hashlib
    return hashlib.sha1((bundle() + bundle('css')).encode()).hexdigest()[:10]

In [ ]:
test_eq(len(version()), 10)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()